# Neural Network Month · Project 2
## Notebook 3 of 3 — Five Folds, Honest Scores

One train/test split can flatter or punish a model by luck. **Stratified 5-fold cross-validation** trains five separate CNNs. Each one is tested on a different fifth of the scans (with the same class mix in every fifth) and trained on the other four-fifths. Every scan is therefore graded exactly once, by a network that never trained on it.

**This notebook:** train the five folds, then look at accuracy, mistakes, per-class behaviour, and *where the network looks*. Four figures. Later notebooks extend the numbered figure series through Figure 10; exported PNGs go in `figures/`.

Set the Colab runtime to a **T4 GPU** first. The full run takes a while, so expect to wait; `SMOKE_TEST = True` below does a fast dry run.

## 1 · Setup, data, model code

In [ ]:
import os, glob, zipfile, hashlib, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedKFold

# One palette for the whole series. Each tumor class keeps its colour in every figure.
P = {
    "glioma":     "#C97B4A",  # clay
    "meningioma": "#2F6F73",  # teal
    "pituitary":  "#D9A441",  # ochre
    "no_tumor":   "#7A9E7E",  # sage
    "ink":        "#4A3F35",  # bark   - text, validation lines
    "paper":      "#F5EFE6",  # sand   - background
    "grid":       "#D8CFC0",  # stone  - gridlines
    "stone":      "#A89F91",  # neutral bars and training lines
    "rust":       "#A5402D",  # warning colour (duplicates, errors)
}
CLASS_COLORS = {"Glioma": P["glioma"], "Meningioma": P["meningioma"],
                "Pituitary": P["pituitary"], "No Tumor": P["no_tumor"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["ink"]])
CAM_CMAP = LinearSegmentedColormap.from_list("cam", [P["pituitary"], P["glioma"], P["rust"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

In [ ]:
# Where is the data? The cell tries, in order: an already-extracted folder, your uploaded
# Kaggle zip, then a Kaggle download. Easiest route: download archive.zip from the Kaggle page
# and upload it to /content (the Colab file panel), or point ZIP_PATH at it in Google Drive.
ZIP_PATH  = "data/archive.zip"  # place the supplied archive here, or change this path
COLAB_ZIP_PATH = "/content/archive.zip"
DATA_DIR  = "mri_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/Train", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(COLAB_ZIP_PATH):
    ZIP_PATH = COLAB_ZIP_PATH
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        path = kagglehub.dataset_download("ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes")
        root = locate_root(path)
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Put the supplied zip at data/archive.zip (or /content/archive.zip in Colab), or add Kaggle credentials.")
print("Data folder:", root)

In [ ]:
# ---- Cleaning settings (the only places we touch the data) ----
KEEP_ONLY_512   = True   # keep only 512x512 scans, so nothing is ever resized
MIN_CLASS_COUNT = 100  # 512x512 No Tumor has only 16 files; it cannot support 5 folds    # a class with fewer images than this after cleaning is dropped
N_FOLDS, SEED   = 5, 42

def index_images(root):
    """One row per image: where it is, its shipped split, class, size, and a fingerprint of its bytes."""
    rows = []
    for p in sorted(glob.glob(f"{root}/*/*/images/*.jpg")):
        rel = os.path.relpath(p, root)
        split, cls = rel.split(os.sep)[:2]
        with Image.open(p) as im:
            w, h = im.size
        with open(p, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        rows.append((rel, split, cls, w, h, md5))
    return pd.DataFrame(rows, columns=["path", "shipped_split", "cls", "w", "h", "md5"])

def clean(raw):
    stages = {"Raw Download": raw}
    df = raw
    if KEEP_ONLY_512:
        df = df[(df.w == 512) & (df.h == 512)]
    stages["512 Pixels Only" if KEEP_ONLY_512 else "All Sizes Kept"] = df

    # How contaminated is the shipped validation set? (measured before removing anything else)
    train_hashes = set(df[df.shipped_split == "Train"].md5)
    val = df[df.shipped_split == "Val"]
    leak = {"val_total": len(val), "val_copies": int(val.md5.isin(train_hashes).sum()),
            "all_total": len(df), "all_redundant": int(len(df) - df.md5.nunique())}

    # Drop any file whose identical twin carries a different label, then keep one copy of each image
    mixed = df.groupby("md5")["cls"].nunique()
    leak["conflicting_labels"] = int((mixed > 1).sum())
    df = df[~df.md5.isin(mixed[mixed > 1].index)]
    df = df.sort_values(["shipped_split", "path"]).drop_duplicates("md5", keep="first")
    stages["Duplicates Removed"] = df

    counts = df.cls.value_counts()
    dropped = {c: int(n) for c, n in counts.items() if n < MIN_CLASS_COUNT}
    df = df[~df.cls.isin(list(dropped))]
    stages["Final Dataset"] = df

    df = df.reset_index(drop=True)
    classes = sorted(df.cls.unique())
    df["label"] = df.cls.map({c: i for i, c in enumerate(classes)})
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, test_idx) in enumerate(skf.split(df, df.label)):
        df.loc[test_idx, "fold"] = k
    return df, stages, leak, dropped, classes

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device)
if device.type == "cpu":
    print("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU. "
          "(SMOKE_TEST = True still works on CPU, just slowly.)")

In [ ]:
SMOKE_TEST = False   # True = quick dry run (20 scans per class, 2 epochs) to check everything works
EPOCHS     = 2 if SMOKE_TEST else 25

if os.path.exists("clean_manifest.csv"):
    manifest = pd.read_csv("clean_manifest.csv")
else:                                   # Notebook 1 was not run in this session: rebuild the same table
    manifest, _, _, _, _ = clean(index_images(root))
if SMOKE_TEST:
    manifest = manifest.groupby("cls", group_keys=False).sample(20, random_state=0).reset_index(drop=True)
classes = sorted(manifest.cls.unique())
n_classes = len(classes)
print(f"{len(manifest)} scans | classes: {classes}")
manifest.groupby("cls").size()

In [ ]:
def load_images(df, root):
    """Read every scan as one 512x512 grey channel. The RGB files are grey-valued copies,
    so one channel keeps all of their information. Nothing else is changed."""
    X = np.empty((len(df), 512, 512), dtype=np.uint8)
    for i, p in enumerate(df.path):
        im = Image.open(os.path.join(root, p)).convert("L")
        if im.size != (512, 512):          # only happens if KEEP_ONLY_512 = False
            im = im.resize((512, 512), Image.BICUBIC)
        X[i] = np.asarray(im)
    return X

t0 = time.time()
X = load_images(manifest, root)
labels = manifest.label.values
folds  = manifest.fold.values
print(f"Loaded {X.shape[0]} scans, {X.nbytes/1e9:.2f} GB, in {time.time()-t0:.0f}s")

In [ ]:
class ResBlock(nn.Module):
    """Two 3x3 convolutions plus a shortcut that lets the signal skip past them."""
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = (nn.Identity() if stride == 1 and c_in == c_out else
                     nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + self.skip(x))

def stage(c_in, c_out, stride, n_blocks):
    return nn.Sequential(ResBlock(c_in, c_out, stride), *[ResBlock(c_out, c_out, 1) for _ in range(n_blocks - 1)])

class BrainCNN(nn.Module):
    """512x512 scan -> learned 4x downsampling -> four residual stages -> average pool -> class scores."""
    def __init__(self, n_classes, widths=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv2d(1, widths[0], 4, 4, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        self.stage1 = stage(widths[0], widths[0], 1, 1)
        self.stage2 = stage(widths[0], widths[1], 2, 2)
        self.stage3 = stage(widths[1], widths[2], 2, 2)
        self.stage4 = stage(widths[2], widths[3], 2, 2)
        self.drop, self.fc = nn.Dropout(dropout), nn.Linear(widths[3], n_classes)

    def features(self, x):                    # the last feature maps, before pooling
        return self.stage4(self.stage3(self.stage2(self.stage1(self.stem(x)))))

    def forward(self, x):
        f = self.features(x)
        return self.fc(self.drop(f.mean(dim=(2, 3))))

In [ ]:
def augment(xb):
    """Light, train-time-only variation (the files on disk are never changed): flip, contrast/brightness, small shift."""
    n, dev = xb.size(0), xb.device
    flip = torch.rand(n, device=dev) < 0.5
    xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
    c = 1 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.3
    b = (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.1
    xb = ((xb - 0.5) * c + 0.5 + b).clamp(0, 1)
    dx, dy = torch.randint(-16, 17, (2,)).tolist()
    return torch.roll(xb, shifts=(dy, dx), dims=(2, 3))

@torch.no_grad()
def predict(model, Xg, idx, bs=64):
    model.eval()
    out = []
    for i in range(0, len(idx), bs):
        xb = Xg[idx[i:i + bs]].unsqueeze(1).float().div(255)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(model(xb).float().cpu())
    return torch.cat(out)

def train_fold(Xg, yg, tr_idx, va_idx, epochs, lr=2e-3, bs=32, use_augment=True, seed=0, verbose=True):
    torch.manual_seed(seed)
    model = BrainCNN(n_classes).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(tr_idx) / bs))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    yv = yg[va_idx].cpu()
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for ep in range(epochs):
        model.train(); t0 = time.time()
        perm = tr_idx[torch.randperm(len(tr_idx), device=tr_idx.device)]
        loss_sum = correct = 0.0
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            xb, yb = Xg[b].unsqueeze(1).float().div(255), yg[b]
            if use_augment:
                xb = augment(xb)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item() * len(b); correct += (logits.argmax(1) == yb).sum().item()
        lv = predict(model, Xg, va_idx)
        hist["train_loss"].append(loss_sum / len(tr_idx)); hist["train_acc"].append(correct / len(tr_idx))
        hist["val_loss"].append(F.cross_entropy(lv, yv).item()); hist["val_acc"].append((lv.argmax(1) == yv).float().mean().item())
        if verbose and (ep % 5 == 4 or ep == epochs - 1 or ep == 0):
            print(f"epoch {ep+1:>2}/{epochs}  train acc {hist['train_acc'][-1]:.3f}  "
                  f"val acc {hist['val_acc'][-1]:.3f}  ({time.time()-t0:.0f}s/epoch)")
    return model, hist, lv

## 2 · Train all five folds

Same network, same settings as Notebook 2, trained from scratch five times. We keep every held-out prediction so we can score the whole dataset at the end.

In [ ]:
Xg = torch.from_numpy(X).to(device)
yg = torch.as_tensor(labels, dtype=torch.long, device=device)

oof_logits = np.zeros((len(manifest), n_classes), dtype=np.float32)
fold_rows, state_dicts = [], []

for k in range(N_FOLDS):
    print(f"--- Fold {k+1} of {N_FOLDS} ---")
    tr_idx = torch.as_tensor(np.where(folds != k)[0], device=device)
    va_np  = np.where(folds == k)[0]
    va_idx = torch.as_tensor(va_np, device=device)

    model, hist, lv = train_fold(Xg, yg, tr_idx, va_idx, epochs=EPOCHS, seed=k)
    oof_logits[va_np] = lv.numpy()
    pred = lv.argmax(1).numpy()
    fold_rows.append({"Fold": k + 1, "Accuracy": (pred == labels[va_np]).mean(),
                      "Macro F1": f1_score(labels[va_np], pred, average="macro")})
    state_dicts.append({n: v.detach().cpu() for n, v in model.state_dict().items()})
    del model
    if device.type == "cuda": torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_rows)
oof_prob = torch.softmax(torch.from_numpy(oof_logits), 1).numpy()
oof_pred = oof_prob.argmax(1)
print("\nPer-fold results:"); display(fold_df.round(3))
print(f"Mean accuracy {fold_df.Accuracy.mean():.3f} +/- {fold_df.Accuracy.std():.3f} | "
      f"overall out-of-fold accuracy {(oof_pred == labels).mean():.3f}")

pd.DataFrame({"path": manifest.path, "true": [classes[i] for i in labels], "predicted": [classes[i] for i in oof_pred],
              "confidence": oof_prob.max(1), "fold": folds + 1}).to_csv("oof_predictions.csv", index=False)

## 3 · Figure 7 — accuracy in each fold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(N_FOLDS); w = 0.38
b1 = ax.bar(x - w/2, fold_df.Accuracy, w, color=P["ink"], label="Accuracy")
b2 = ax.bar(x + w/2, fold_df["Macro F1"], w, color=P["stone"], label="Macro F1")
ax.bar_label(b1, fmt="%.3f", padding=2, fontsize=9); ax.bar_label(b2, fmt="%.3f", padding=2, fontsize=9)
ax.axhline(fold_df.Accuracy.mean(), color=P["rust"], linestyle="--", linewidth=1.6, label="Mean accuracy")
lo = min(fold_df.Accuracy.min(), fold_df["Macro F1"].min())
ax.set_ylim(max(0, lo - 0.08), 1.03)
ax.set_xticks(x); ax.set_xticklabels([f"Fold {k+1}" for k in range(N_FOLDS)])
ax.set_ylabel("Score"); ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_title("Five-Fold Accuracy Scores")
fig.tight_layout(); save_fig(fig, "07_fold_scores"); plt.show()

**How to read it.** The spread across folds tells you how stable the result is. *Macro F1* averages each class's score with equal weight, so it drops if the network neglects a smaller class even while overall accuracy looks fine. Remember the caveat from Notebook 1: near-copies of a scan can survive on both sides of a fold, so treat these numbers as an upper-leaning estimate, not a clinical guarantee.

## 4 · Figure 8 — where the mistakes go

In [ ]:
cm = confusion_matrix(labels, oof_pred, labels=range(n_classes))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.imshow(cmn, cmap=SEQ_CMAP, vmin=0, vmax=1); ax.grid(False)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center",
                color="white" if cmn[i, j] > 0.5 else P["ink"], fontsize=12, fontweight="bold")
ax.set_xticks(range(n_classes)); ax.set_xticklabels(classes); ax.set_yticks(range(n_classes)); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("Out-of-Fold Confusion Matrix")
fig.tight_layout(); save_fig(fig, "08_confusion_matrix"); plt.show()

Each row is a true class; the diagonal is correct. Any large off-diagonal cell is a specific confusion worth discussing: it tells you which pair of tumor types the network finds hardest to separate.

## 5 · Figure 9 — precision and recall by class

In [ ]:
prec, rec, _, _ = precision_recall_fscore_support(labels, oof_pred, labels=range(n_classes), zero_division=0)
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(n_classes); w = 0.38
for i, c in enumerate(classes):
    ax.bar(i - w/2, rec[i], w, color=CLASS_COLORS[c])
    ax.bar(i + w/2, prec[i], w, color=CLASS_COLORS[c], alpha=0.55, hatch="//", edgecolor=P["paper"])
    ax.text(i - w/2, rec[i] + 0.006, f"{rec[i]:.2f}", ha="center", fontsize=9)
    ax.text(i + w/2, prec[i] + 0.006, f"{prec[i]:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(classes); ax.set_ylim(max(0, min(prec.min(), rec.min()) - 0.1), 1.04)
ax.set_ylabel("Score")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=P["stone"], label="Recall (solid)"), Patch(facecolor=P["stone"], alpha=0.55, hatch="//", label="Precision (hatched)")],
          frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_title("Per-Class Precision and Recall")
fig.tight_layout(); save_fig(fig, "09_precision_recall"); plt.show()

**Recall** asks: of the scans that truly are this class, how many did we catch? **Precision** asks: of the scans we *called* this class, how many were right? A class can have high recall and low precision if the network over-uses it.

## 6 · Figure 10 — where the model looks

Our network ends in global average pooling and one linear layer, which allows a neat trick called a **class activation map (CAM)**: multiply each of the 256 final feature maps by the weight the final layer gives it for the predicted class and add them up. The result is a 16×16 heat map of the regions that pushed the decision toward that class, stretched over the scan.

We show correctly classified scans, using for each scan the network that **never trained on it** (its own fold's model).

In [ ]:
def load_model(k):
    m = BrainCNN(n_classes).to(device); m.load_state_dict(state_dicts[k]); return m.eval()
fold_models = {}

@torch.no_grad()
def cam_for(model, x, cls_idx):
    f = model.features(x)[0]
    cam = torch.einsum("c,chw->hw", model.fc.weight[cls_idx], f).relu()
    cam = F.interpolate(cam[None, None], size=(512, 512), mode="bilinear", align_corners=False)[0, 0]
    return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy()

rng = np.random.default_rng(11)
N_COLS = 4
fig, axes = plt.subplots(n_classes, N_COLS, figsize=(3.3 * N_COLS, 3.3 * n_classes))
axes = np.atleast_2d(axes)
for r, c in enumerate(classes):
    ok = np.where((labels == c) & (oof_pred == c))[0]
    pick = rng.choice(ok, min(N_COLS, len(ok)), replace=False) if len(ok) else []
    for a in axes[r]: a.axis("off")
    axes[r][0].text(-0.06, 0.5, c, transform=axes[r][0].transAxes, rotation=90, ha="right", va="center",
                    fontsize=13, fontweight="bold", color=CLASS_COLORS[c])
    for a, i in zip(axes[r], pick):
        k = int(folds[i])
        if k not in fold_models:
            fold_models[k] = load_model(k)
        x = Xg[i][None, None].float().div(255)
        cam = cam_for(fold_models[k], x, c)
        rgba = CAM_CMAP(cam); rgba[..., 3] = cam * 0.8
        a.imshow(X[i], cmap="gray"); a.imshow(rgba)
        a.set_title(f"{c} {oof_prob[i, c]:.0%}", fontsize=11, color=CLASS_COLORS[c])
fig.suptitle("Where The Model Looks", y=1.0)
fig.tight_layout(); save_fig(fig, "10_where_model_looks"); plt.show()

**How to read it.** Warm, opaque patches mark where the network found evidence for its answer. If those patches sit on or around the visible abnormality, that is reassuring. If they sit on the skull edge, the background, or the same part of every scan of a class, the network may be using a **shortcut** (such as scan angle or contrast) instead of the tumor.

**What this cannot do:** a heat map shows what influenced the network, not that the network "understands" tumors, and attention that lands on a tumor is not proof of clinical reasoning. Treat it as a sanity check, not a diagnosis.

## What we showed

- A residual CNN on **full-resolution** scans, with only necessary cleaning (size filter, exact-duplicate removal) and no normalisation or hand-picking.
- **Stratified 5-fold cross-validation** over exact-deduplicated 512×512 files. Exact duplicate removal prevents byte-identical copies from crossing folds; near duplicates and patient overlap cannot be ruled out.
- Where the model succeeds, where it confuses classes, and (with CAMs) what it appears to rely on.

This is an educational demonstration on a public dataset. It is **not** a diagnostic tool.

**Coming up:** the final project of the month puts real neuroscience beside everything we have built.

In [ ]:
# Optional: download figures and predictions
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb3_figures", "zip", "figures")
    files.download("nb3_figures.zip"); files.download("oof_predictions.csv")
except ImportError:
    print("Figures are saved in the 'figures' folder.")